In [73]:
# step 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [74]:
# step 2: Load Grocery Dataset

transactions = []
try:
    with open('groceries.csv', 'r') as f:
        for line in f:
            # Clean the line, split by comma, and remove empty items
            items = [item.strip() for item in line.strip().split(',') if item.strip()]
            if items:
                transactions.append(items)
    
    # Define variables for later steps
    baskets = transactions
    transaction_lengths = [len(t) for t in transactions]

    print(f"✅ Dataset loaded successfully")
    print(f"Total Transactions: {len(transactions)}")
    print(f"First Transaction: {transactions[0]}")

except FileNotFoundError:
    print("❌ Error: 'groceries.csv' not found. Please ensure the file is uploaded.")

✅ Dataset loaded successfully
Total Transactions: 9835
First Transaction: ['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups']


In [ ]:
# step 3: Data Validation

transaction_lengths = [len(t) for t in transactions]

print("Transaction length statistics:")
print(f"Min items: {np.min(transaction_lengths)}")
print(f"Max items: {np.max(transaction_lengths)}")
print(f"Average items: {np.mean(transaction_lengths):.2f}")

unique_items = set(item for t in transactions for item in t)
print(f"Total unique items: {len(unique_items)}")


Missing values per column:
Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

Duplicate rows: 0

Invalid total calculations: 0

Data validation completed


In [76]:
# Cell 4: Transaction Structure Analysis
print("Transaction structure:")
print(f"Number of transactions: {len(transactions)}")
print(f"Average basket size: {np.mean(transaction_lengths):.2f}")
print("Data is already in market basket format")

Transaction structure:
Number of transactions: 9835
Average basket size: 4.41
Data is already in market basket format


In [77]:
# Cell 5: One-Hot Encoding
te = TransactionEncoder()
encoded_array = te.fit(transactions).transform(transactions)

encoded_df = pd.DataFrame(encoded_array, columns=te.columns_)

print("Encoded dataset shape:", encoded_df.shape)
encoded_df.head()


Encoded dataset shape: (9835, 169)


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [78]:
# Cell 6: Apriori Algorithm
min_support = 0.01

frequent_itemsets = apriori(
    encoded_df,
    min_support=min_support,
    use_colnames=True,
    max_len=3
)

frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)

print(f"Frequent itemsets found: {len(frequent_itemsets)}")
frequent_itemsets.head(10)


Frequent itemsets found: 333


,support,itemsets,length
86,0.255516,(whole milk),1
55,0.193493,(other vegetables),1
66,0.183935,(rolls/buns),1
75,0.174377,(soda),1
87,0.139502,(yogurt),1
6,0.110524,(bottled water),1
67,0.108998,(root vegetables),1
81,0.104931,(tropical fruit),1
73,0.098526,(shopping bags),1
70,0.093950,(sausage),1


In [79]:
# step 7: Association Rules
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.1
)

rules = rules.sort_values("lift", ascending=False)

print(f"Association rules generated: {len(rules)}")
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)


Association rules generated: 460


,antecedents,consequents,support,confidence,lift
453,(curd),"(whole milk, yogurt)",0.010066,0.188931,3.372304
452,"(whole milk, yogurt)",(curd),0.010066,0.179673,3.372304
416,"(citrus fruit, other vegetables)",(root vegetables),0.010371,0.359155,3.295045
436,"(yogurt, other vegetables)",(whipped/sour cream),0.010168,0.234192,3.267062
437,(whipped/sour cream),"(yogurt, other vegetables)",0.010168,0.141844,3.267062
302,"(other vegetables, tropical fruit)",(root vegetables),0.012303,0.342776,3.144780
303,(root vegetables),"(other vegetables, tropical fruit)",0.012303,0.112873,3.144780
162,(root vegetables),(beef),0.017387,0.159515,3.040367
163,(beef),(root vegetables),0.017387,0.331395,3.040367
415,"(root vegetables, citrus fruit)",(other vegetables),0.010371,0.586207,3.029608


In [80]:
# Cell 8: Rule Interpretation
for i, rule in rules.head(5).iterrows():
    print(f"{set(rule['antecedents'])} → {set(rule['consequents'])}")
    print(f"Support: {rule['support']:.3f}")
    print(f"Confidence: {rule['confidence']:.3f}")
    print(f"Lift: {rule['lift']:.3f}")
    print("-" * 40)


{'curd'} → {'whole milk', 'yogurt'}
Support: 0.010
Confidence: 0.189
Lift: 3.372
----------------------------------------
{'whole milk', 'yogurt'} → {'curd'}
Support: 0.010
Confidence: 0.180
Lift: 3.372
----------------------------------------
{'citrus fruit', 'other vegetables'} → {'root vegetables'}
Support: 0.010
Confidence: 0.359
Lift: 3.295
----------------------------------------
{'yogurt', 'other vegetables'} → {'whipped/sour cream'}
Support: 0.010
Confidence: 0.234
Lift: 3.267
----------------------------------------
{'whipped/sour cream'} → {'yogurt', 'other vegetables'}
Support: 0.010
Confidence: 0.142
Lift: 3.267
----------------------------------------


In [84]:
# step 10: Rule Testing (Updated for real item names)

def recommend_items(input_basket, rules_df, top_n=3):
    if rules_df.empty:
        return "No rules available."

    # Filter rules where the input is part of the antecedents
    # Note: input_basket should be a set, e.g., {"whole milk"}
    matches = rules_df[rules_df['antecedents'].apply(lambda x: input_basket.issubset(x))]
    
    if matches.empty:
        return "No specific associations found for this item."

    # Sort by Lift to get the strongest connections
    matches = matches.sort_values(by='lift', ascending=False)
    
    recommendations = []
    for _, rule in matches.iterrows():
        for item in rule['consequents']:
            if item not in input_basket and item not in recommendations:
                recommendations.append(item)
    
    return recommendations[:top_n]

# --- TEST WITH THE CORRECT NAME ---
# In groceries.csv, it is 'whole milk', not 'milk'
test_item = {"pip fruit"} 
output = recommend_items(test_item, rules)

print(f"🛒 Input Basket: {test_item}")
print(f"🥇 Recommended Items: {output}")

🛒 Input Basket: {'pip fruit'}
🥇 Recommended Items: ['tropical fruit', 'whole milk', 'other vegetables']
